In [ ]:
!pip install grapheme indic-nlp-library


In [ ]:
!pip install -U transformers accelerate


  Using cached transformers-5.1.0-py3-none-any.whl.metadata (31 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 73.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
!pip install transformers datasets scikit-learn grapheme

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving trainV2.csv to trainV2.csv


In [ ]:
import pandas as pd
df = pd.read_csv("trainV2.csv")
df.head()


,Text,Class
0,நான் கூட உன்னை வெகுளியான பொண்ணு&#39;னு நெனச்சி...,Non-Abusive
1,உன் போட்டோவை டாய்லெட்டுக்கு மாற்றினார்கள் அசிங...,Abusive
2,கண்டா வரச்சொல்லுங்க கார்த்திய திவ்யாவோட சேர்த்...,Non-Abusive
3,ஒன்னோட சைசுக்கு நீயே ஒரு நாளக்கி 5கிலோ ஆய் போவ...,Abusive
4,ரெண்டும் மிக பெரிய வெடிகுண்டு இவங்கள எதுக்கு ஷ...,Non-Abusive


In [ ]:
import re, html, unicodedata
import grapheme

def grapheme_preprocess(text):
    text = str(text)

    # decode html
    text = html.unescape(text)

    # normalize unicode (VERY IMPORTANT)
    text = unicodedata.normalize("NFC", text)

    # remove urls
    text = re.sub(r'http\S+|www\S+', '', text)

    # extract grapheme clusters
    clusters = list(grapheme.graphemes(text))

    # rejoin normalized graphemes
    text = "".join(clusters)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text


In [ ]:
df["grapheme_text"] = df["Text"].apply(grapheme_preprocess)
df["clean_text"] = df["Text"]  # baseline version


In [ ]:
TEXT_COL = "grapheme_text"
#TEXT_COL = "clean_text" for baseline version


In [ ]:
print(df.columns)


Index(['Text', 'Class', 'grapheme_text', 'clean_text', 'label'], dtype='object')


In [ ]:
df["label"] = df["Class"].map({
    "Non-Abusive":0,
    "Abusive":1
})


In [ ]:
# normalize label text
df["Class"] = df["Class"].astype(str).str.strip().str.lower()

print("Unique labels after cleaning:", df["Class"].unique())

# map to numeric
df["label"] = df["Class"].map({
    "abusive":1,
    "non-abusive":0,
    "non abusive":0
})

# check if any missing
print("NaN labels count:", df["label"].isna().sum())


Unique labels after cleaning: ['non-abusive' 'abusive']
NaN labels count: 0


In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df[[TEXT_COL,"label"]],
    test_size=0.1,
    stratify=df["label"],
    random_state=42
)


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "ai4bharat/IndicBERTv2-MLM-only"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.75M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ai4bharat/IndicBERTv2-MLM-only
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if y

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df.rename(columns={TEXT_COL:"text"})
)
val_dataset = Dataset.from_pandas(
    val_df.rename(columns={TEXT_COL:"text"})
)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

train_dataset.set_format("torch", columns=["input_ids","attention_mask","label"])
val_dataset.set_format("torch", columns=["input_ids","attention_mask","label"])


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Map:   0%|          | 0/3286 [00:00<?, ? examples/s]

Map:   0%|          | 0/366 [00:00<?, ? examples/s]

In [ ]:
from transformers import Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,   # NEW change
    compute_metrics=compute_metrics
)

trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 3}.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.457469,0.792350,0.781609
2,No log,0.449682,0.789617,0.796834
3,0.460234,0.445206,0.825137,0.824176
4,0.460234,0.507337,0.819672,0.805882
5,0.201462,0.553783,0.830601,0.821839


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1030, training_loss=0.3259553575978696, metrics={'train_runtime': 655.6026, 'train_samples_per_second': 25.061, 'train_steps_per_second': 1.571, 'total_flos': 1080728659891200.0, 'train_loss': 0.3259553575978696, 'epoch': 5.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.4450267553329468,
 'eval_accuracy': 0.8278688524590164,
 'eval_f1': 0.8264462809917356,
 'eval_runtime': 2.6958,
 'eval_samples_per_second': 135.767,
 'eval_steps_per_second': 8.532,
 'epoch': 5.0}

In [ ]:
trainer.save_model("best_model_grapheme")
tokenizer.save_pretrained("best_model_grapheme")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('best_model_grapheme/tokenizer_config.json',
 'best_model_grapheme/tokenizer.json')

In [ ]:
import shutil
shutil.make_archive("IndicBert_grapheme_best_model", 'zip', "best_model_grapheme")

'/content/IndicBert_grapheme_best_model.zip'

In [ ]:
from google.colab import files
files.download("IndicBert_grapheme_best_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>